# Assignment 1B — Part B (Instruction Fine-Tuning with QLoRA)

This notebook implements **Part B1–B3** using only active workspace assets (no `old/` usage):
- B1: Instruction dataset creation (minimum 100 pairs, JSONL, 80/20 split)
- B2: QLoRA fine-tuning with three LoRA adapter configurations (A, B, C)
- B3: Evaluation against the same 3 domain prompts used in Part A baseline

> Safe defaults: B1 runs by default; B2 and B3 are opt-in because they are compute-intensive.

In [ ]:
# 1) Set Up Environment and Imports
from __future__ import annotations

import inspect
import csv
import json
import math
import os
import random
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable, Iterable

import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

ROOT = Path.cwd()
if not (ROOT / 'clean_corpus').exists():
    candidate = ROOT / 'LLM_Assignment_Medical_Solution'
    if (candidate / 'clean_corpus').exists():
        ROOT = candidate

CLEAN_CORPUS = ROOT / 'clean_corpus'
OUTPUTS = ROOT / 'outputs' / 'assignment1b_partb'
OUTPUTS.mkdir(parents=True, exist_ok=True)

INSTRUCTION_JSONL = OUTPUTS / 'instruction_dataset.jsonl'
TRAIN_JSONL = OUTPUTS / 'instruction_train.jsonl'
EVAL_JSONL = OUTPUTS / 'instruction_eval.jsonl'
COUNTS_JSON = OUTPUTS / 'dataset_counts.json'
ADAPTER_ROOT = OUTPUTS / 'qlora_adapters'
COMPARISON_CSV = OUTPUTS / 'adapter_comparison.csv'
COMPARISON_JSON = OUTPUTS / 'adapter_comparison.json'

BASELINE_PATH = ROOT / 'outputs' / 'baseline_outputs.json'

RUN_B1 = True
RUN_B2 = False
RUN_B3 = False

MODEL_ID = 'microsoft/biogpt-large'
# Pin the revision that contains the published safetensors checkpoint.
MODEL_REVISION = '4585354702bf941a07727e316182421fea0051fe'
MAX_PAIRS = 120
MIN_REQUIRED_PAIRS = 100
TRAIN_FRACTION = 0.8

DOMAIN_PROMPTS = [
    'Explain why repeated blood-pressure measurements are useful in hypertension.',
    'What is the relationship between insulin resistance and type 2 diabetes?',
    'Why is antimicrobial stewardship important?'
]

SYNTHETIC_PROMPT_TEMPLATE = (
    "Read the text below and generate 10 instruction-response pairs in JSON format based ONLY on this text. "
    "Each entry must have instruction and response keys."
)

def disable_optional_scipy_backend():
    """Prevent Transformers from importing unused, broken SciPy optional losses."""
    import transformers.utils as _transformers_utils
    import transformers.utils.import_utils as _import_utils
    _no_scipy = lambda: False
    _import_utils._scipy_available = False
    _import_utils.is_scipy_available = _no_scipy
    _transformers_utils.is_scipy_available = _no_scipy

print({'root': str(ROOT), 'model': MODEL_ID, 'outputs': str(OUTPUTS), 'RUN_B1': RUN_B1, 'RUN_B2': RUN_B2, 'RUN_B3': RUN_B3})

## 2) Encode Assignment-1B Requirements as Assertions

This cell turns the Part-B acceptance criteria into executable checks so failures are explicit and early.

In [ ]:
REQUIREMENTS = {
    'clean_corpus_exists': CLEAN_CORPUS.exists(),
    'clean_txt_files': len(list(CLEAN_CORPUS.glob('*.txt'))),
    'min_pairs_required': MIN_REQUIRED_PAIRS,
    'split_ratio': (TRAIN_FRACTION, round(1 - TRAIN_FRACTION, 2)),
    'adapter_configs_required': ['A', 'B', 'C'],
    'compare_prompt_count': 3,
}

assert REQUIREMENTS['clean_corpus_exists'], f'Missing clean corpus directory: {CLEAN_CORPUS}'
assert REQUIREMENTS['clean_txt_files'] > 0, 'No cleaned .txt files found in clean_corpus/.'
assert MIN_REQUIRED_PAIRS >= 100, 'Part B1 requires at least 100 instruction-response pairs.'
assert math.isclose(TRAIN_FRACTION, 0.8, rel_tol=0, abs_tol=1e-9), 'Part B1 split must be 80/20.'
assert len(DOMAIN_PROMPTS) == REQUIREMENTS['compare_prompt_count'], 'Part B3 requires exactly the same 3 domain prompts.'

print('Assignment-1B Part-B requirement checks passed:')
print(json.dumps(REQUIREMENTS, indent=2))

## 3) Create/Verify Module and Function Interfaces

Define interfaces first and assert signature completeness before execution.

In [ ]:
@dataclass(frozen=True)
class AdapterSpec:
    name: str
    rank: int
    alpha: int
    target_modules: tuple[str, ...]


def read_clean_documents(clean_dir: Path) -> list[tuple[str, str]]:
    ...


def build_instruction_pairs(docs: list[tuple[str, str]], max_pairs: int, min_chars: int = 80) -> list[dict[str, str]]:
    ...


def split_pairs(pairs: list[dict[str, str]], train_fraction: float, seed: int) -> tuple[list[dict[str, str]], list[dict[str, str]]]:
    ...


def write_jsonl(rows: list[dict[str, str]], path: Path) -> None:
    ...


def prepare_b1_dataset(clean_dir: Path, output_dir: Path, max_pairs: int, train_fraction: float, seed: int) -> dict[str, Any]:
    ...


def train_qlora_adapter(model_id: str, train_rows: list[dict[str, str]], eval_rows: list[dict[str, str]], adapter_spec: AdapterSpec, output_dir: Path, max_steps: int = 60) -> Path:
    ...


def load_baseline_outputs(baseline_path: Path) -> list[dict[str, str]]:
    ...


def compare_adapters(model_id: str, adapter_dirs: dict[str, Path], prompts: list[str], baseline_outputs: list[dict[str, str]]) -> list[dict[str, str]]:
    ...


REQUIRED_SIGNATURES: dict[str, list[str]] = {
    'read_clean_documents': ['clean_dir'],
    'build_instruction_pairs': ['docs', 'max_pairs', 'min_chars'],
    'split_pairs': ['pairs', 'train_fraction', 'seed'],
    'write_jsonl': ['rows', 'path'],
    'prepare_b1_dataset': ['clean_dir', 'output_dir', 'max_pairs', 'train_fraction', 'seed'],
    'train_qlora_adapter': ['model_id', 'train_rows', 'eval_rows', 'adapter_spec', 'output_dir', 'max_steps'],
    'load_baseline_outputs': ['baseline_path'],
    'compare_adapters': ['model_id', 'adapter_dirs', 'prompts', 'baseline_outputs'],
}

for fn_name, expected in REQUIRED_SIGNATURES.items():
    params = list(inspect.signature(globals()[fn_name]).parameters.keys())
    assert params == expected, f'Signature mismatch for {fn_name}: expected {expected}, got {params}'

print('Interface signature checks passed for Part-B pipeline functions.')

## 4) Implement Core Assignment-1B Logic

Implement B1/B2/B3 functions and run lightweight inline checks.

In [ ]:
from collections import defaultdict


def _normalize_whitespace(text: str) -> str:
    return re.sub(r'\s+', ' ', text).strip()


def _sentences(text: str) -> list[str]:
    chunks = re.split(r'(?<=[\.!?])\s+', _normalize_whitespace(text))
    return [chunk.strip() for chunk in chunks if len(chunk.strip()) > 20]


def _safe_response(sentence: str, max_chars: int = 450) -> str:
    sentence = sentence.strip()
    if len(sentence) <= max_chars:
        return sentence
    clipped = sentence[:max_chars].rsplit(' ', 1)[0].strip()
    return clipped + '...'


def read_clean_documents(clean_dir: Path) -> list[tuple[str, str]]:
    if not clean_dir.exists():
        raise FileNotFoundError(f'Clean corpus directory not found: {clean_dir}')
    files = sorted(clean_dir.glob('*.txt'))
    if not files:
        raise ValueError(f'No .txt files found in clean corpus: {clean_dir}')
    rows: list[tuple[str, str]] = []
    for path in files:
        text = path.read_text(encoding='utf-8', errors='ignore').strip()
        if text:
            rows.append((path.name, text))
    if not rows:
        raise ValueError('All cleaned files were empty.')
    return rows


def build_instruction_pairs(docs: list[tuple[str, str]], max_pairs: int, min_chars: int = 80) -> list[dict[str, str]]:
    if max_pairs <= 0:
        raise ValueError('max_pairs must be > 0')
    templates = [
        'Summarize this clinical point from domain text: {topic}',
        'Explain this medical statement in clear terms: {topic}',
        'What does the following domain evidence imply? {topic}',
        'Provide a concise evidence-based explanation: {topic}',
        'Based only on the corpus, clarify: {topic}',
        'What is the clinical significance of: {topic}',
        'Interpret this domain observation: {topic}',
    ]
    followup_templates = [
        'Answer this question using only domain evidence: {topic}',
        'What should a learner understand from this statement? {topic}',
    ]

    topic_counts: defaultdict[str, int] = defaultdict(int)
    pairs: list[dict[str, str]] = []

    for file_name, text in docs:
        for sentence in _sentences(text):
            if len(sentence) < min_chars:
                continue
            topic = sentence[:120].strip()

            prompt_template = templates[topic_counts[file_name] % len(templates)]
            topic_counts[file_name] += 1
            pairs.append({
                'instruction': prompt_template.format(topic=topic),
                'response': _safe_response(sentence),
                'source_file': file_name,
            })

            alt_template = followup_templates[topic_counts[file_name] % len(followup_templates)]
            pairs.append({
                'instruction': alt_template.format(topic=topic),
                'response': _safe_response(sentence),
                'source_file': file_name,
            })

            if len(pairs) >= max_pairs * 2:
                break
        if len(pairs) >= max_pairs * 2:
            break

    deduped: list[dict[str, str]] = []
    seen = set()
    for row in pairs:
        key = (_normalize_whitespace(row['instruction']).lower(), _normalize_whitespace(row['response']).lower())
        if key in seen:
            continue
        seen.add(key)
        deduped.append(row)
        if len(deduped) >= max_pairs:
            break

    if len(deduped) < MIN_REQUIRED_PAIRS:
        raise ValueError(
            f'Only {len(deduped)} unique instruction-response pairs created; '
            f'increase corpus size or MAX_PAIRS to satisfy minimum {MIN_REQUIRED_PAIRS}.'
        )
    return deduped


def split_pairs(pairs: list[dict[str, str]], train_fraction: float, seed: int) -> tuple[list[dict[str, str]], list[dict[str, str]]]:
    if not (0 < train_fraction < 1):
        raise ValueError('train_fraction must be between 0 and 1')
    if len(pairs) < 2:
        raise ValueError('At least two pairs are required to split train/eval')
    rng = random.Random(seed)
    shuffled = pairs.copy()
    rng.shuffle(shuffled)
    split_index = max(1, min(len(shuffled) - 1, int(round(len(shuffled) * train_fraction))))
    train_rows = shuffled[:split_index]
    eval_rows = shuffled[split_index:]
    if not train_rows or not eval_rows:
        raise ValueError('Split resulted in empty train or eval set')
    return train_rows, eval_rows


def write_jsonl(rows: list[dict[str, str]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as fp:
        for row in rows:
            payload = {'instruction': row['instruction'], 'response': row['response']}
            fp.write(json.dumps(payload, ensure_ascii=False) + '\n')


def prepare_b1_dataset(clean_dir: Path, output_dir: Path, max_pairs: int, train_fraction: float, seed: int) -> dict[str, Any]:
    docs = read_clean_documents(clean_dir)
    pairs = build_instruction_pairs(docs, max_pairs=max_pairs)
    write_jsonl(pairs, output_dir / 'instruction_dataset.jsonl')
    train_rows, eval_rows = split_pairs(pairs, train_fraction=train_fraction, seed=seed)
    write_jsonl(train_rows, output_dir / 'instruction_train.jsonl')
    write_jsonl(eval_rows, output_dir / 'instruction_eval.jsonl')

    counts = {
        'documents_used': len(docs),
        'total_pairs': len(pairs),
        'train_pairs': len(train_rows),
        'eval_pairs': len(eval_rows),
        'train_fraction': train_fraction,
        'eval_fraction': 1 - train_fraction,
        'synthetic_prompt_template': SYNTHETIC_PROMPT_TEMPLATE,
    }
    (output_dir / 'dataset_counts.json').write_text(json.dumps(counts, indent=2), encoding='utf-8')
    return counts


def _format_for_sft(tokenizer, instruction: str, response: str) -> str:
    messages = [
        {'role': 'user', 'content': instruction},
        {'role': 'assistant', 'content': response},
    ]
    if hasattr(tokenizer, 'apply_chat_template'):
        try:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        except Exception:
            pass
    return f'User: {instruction}\nAssistant: {response}'


def _build_adapter_specs() -> list[AdapterSpec]:
    return [
        AdapterSpec(name='A', rank=8, alpha=16, target_modules=('q_proj', 'v_proj')),
        AdapterSpec(name='B', rank=16, alpha=32, target_modules=('q_proj', 'v_proj')),
        AdapterSpec(name='C', rank=32, alpha=32, target_modules=('q_proj', 'v_proj', 'o_proj')),
    ]


def _rows_to_hf_dataset(rows: list[dict[str, str]], tokenizer):
    from datasets import Dataset
    formatted = [{'text': _format_for_sft(tokenizer, r['instruction'], r['response'])} for r in rows]
    return Dataset.from_list(formatted)


def train_qlora_adapter(model_id: str, train_rows: list[dict[str, str]], eval_rows: list[dict[str, str]], adapter_spec: AdapterSpec, output_dir: Path, max_steps: int = 60) -> Path:
    import torch
    disable_optional_scipy_backend()
    from peft import LoraConfig, get_peft_model
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
    from trl import SFTTrainer

    if max_steps <= 0:
        raise ValueError('max_steps must be > 0')

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=MODEL_REVISION)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        revision=MODEL_REVISION,
        use_safetensors=True,
        quantization_config=quant_config,
        device_map='auto',
    )

    peft_config = LoraConfig(
        r=adapter_spec.rank,
        lora_alpha=adapter_spec.alpha,
        lora_dropout=0.05,
        bias='none',
        task_type='CAUSAL_LM',
        target_modules=list(adapter_spec.target_modules),
    )
    model = get_peft_model(model, peft_config)

    train_ds = _rows_to_hf_dataset(train_rows, tokenizer)
    eval_ds = _rows_to_hf_dataset(eval_rows, tokenizer)

    args = TrainingArguments(
        output_dir=str(output_dir),
        max_steps=max_steps,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        logging_steps=5,
        save_steps=max_steps,
        save_total_limit=1,
        evaluation_strategy='steps',
        eval_steps=max(10, max_steps // 3),
        bf16=torch.cuda.is_available(),
        report_to='none',
        remove_unused_columns=False,
    )

    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        dataset_text_field='text',
        max_seq_length=1024,
        tokenizer=tokenizer,
    )
    trainer.train()
    output_dir.mkdir(parents=True, exist_ok=True)
    trainer.model.save_pretrained(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))
    return output_dir


def _load_generate_pipeline(model_id: str, adapter_dir: Path | None = None):
    import torch
    disable_optional_scipy_backend()
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=MODEL_REVISION)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    base_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        revision=MODEL_REVISION,
        use_safetensors=True,
        dtype=dtype,
        device_map='auto' if torch.cuda.is_available() else None,
    )
    if not torch.cuda.is_available():
        base_model = base_model.to('cpu')

    model = PeftModel.from_pretrained(base_model, str(adapter_dir)) if adapter_dir is not None else base_model
    return model, tokenizer


def _generate_text(model, tokenizer, prompt: str, max_new_tokens: int = 80) -> str:
    import torch
    model.eval()
    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors='pt')
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


def load_baseline_outputs(baseline_path: Path) -> list[dict[str, str]]:
    if not baseline_path.exists():
        raise FileNotFoundError(
            f'Baseline outputs not found at {baseline_path}. Run Part A Step 2 first to produce baseline outputs.'
        )
    rows = json.loads(baseline_path.read_text(encoding='utf-8'))
    if not isinstance(rows, list) or len(rows) < 3:
        raise ValueError('Baseline outputs must be a list with at least 3 rows.')
    return rows


def compare_adapters(model_id: str, adapter_dirs: dict[str, Path], prompts: list[str], baseline_outputs: list[dict[str, str]]) -> list[dict[str, str]]:
    if len(prompts) != 3:
        raise ValueError('Part B3 requires exactly 3 prompts.')
    if len(baseline_outputs) < 3:
        raise ValueError('At least 3 baseline outputs are required.')

    records: list[dict[str, str]] = []
    baseline_by_prompt = {row.get('prompt', ''): row.get('generated_text', '') for row in baseline_outputs}

    for prompt in prompts:
        row = {'prompt': prompt, 'baseline_output': baseline_by_prompt.get(prompt, '')}
        for name, adapter_dir in adapter_dirs.items():
            if not adapter_dir.exists():
                row[f'adapter_{name}_output'] = '[adapter not trained]'
                continue
            model, tokenizer = _load_generate_pipeline(model_id, adapter_dir)
            row[f'adapter_{name}_output'] = _generate_text(model, tokenizer, prompt)
        records.append(row)

    return records


inline_docs = read_clean_documents(CLEAN_CORPUS)
assert len(inline_docs) > 0
inline_pairs = build_instruction_pairs(inline_docs, max_pairs=MAX_PAIRS)
assert len(inline_pairs) >= MIN_REQUIRED_PAIRS
print({'inline_document_count': len(inline_docs), 'inline_pair_count': len(inline_pairs)})

## 5) Add Input Validation and Edge-Case Handling

Validate boundary inputs and provide explicit errors for invalid usage.

In [ ]:
def expect_raises(fn: Callable, expected_exception: type[BaseException], *args, **kwargs) -> str:
    try:
        fn(*args, **kwargs)
    except expected_exception as exc:
        return str(exc)
    except Exception as exc:
        raise AssertionError(f'Expected {expected_exception.__name__}, got {type(exc).__name__}: {exc}') from exc
    raise AssertionError(f'Expected {expected_exception.__name__} but no exception was raised')


assert 'max_pairs must be > 0' in expect_raises(build_instruction_pairs, ValueError, [('a.txt', 'text' * 100)], 0)
assert 'train_fraction must be between 0 and 1' in expect_raises(split_pairs, ValueError, [{'instruction': 'x', 'response': 'y'}] * 2, 1.2, 42)
assert 'At least two pairs are required' in expect_raises(split_pairs, ValueError, [{'instruction': 'x', 'response': 'y'}], 0.8, 42)

print('Edge-case handling checks passed.')

## 6) Write Unit Tests for Required Behaviors

Focused tests cover normal behavior, edge cases, and failure paths aligned with Part-B criteria.

In [ ]:
import unittest


class TestAssignment1BPartB(unittest.TestCase):
    def setUp(self):
        doc1_sentences = [
            f'Hypertension evidence note {i}: repeated blood pressure measurements improve diagnostic reliability and guide treatment adjustments across follow-up visits.'
            for i in range(1, 51)
        ]
        doc2_sentences = [
            f'Diabetes mechanism note {i}: insulin resistance reduces cellular glucose uptake and contributes to type 2 diabetes progression through metabolic dysregulation.'
            for i in range(1, 51)
        ]
        doc3_sentences = [
            f'Stewardship principle {i}: antimicrobial stewardship reduces unnecessary antibiotic exposure, slows resistance emergence, and improves patient safety outcomes.'
            for i in range(1, 51)
        ]
        self.docs = [
            ('doc1.txt', ' '.join(doc1_sentences)),
            ('doc2.txt', ' '.join(doc2_sentences)),
            ('doc3.txt', ' '.join(doc3_sentences)),
        ]

    def test_build_instruction_pairs_minimum(self):
        pairs = build_instruction_pairs(self.docs, max_pairs=110, min_chars=40)
        self.assertGreaterEqual(len(pairs), 100)
        self.assertIn('instruction', pairs[0])
        self.assertIn('response', pairs[0])

    def test_split_ratio_non_empty(self):
        pairs = build_instruction_pairs(self.docs, max_pairs=110, min_chars=40)
        train_rows, eval_rows = split_pairs(pairs, train_fraction=0.8, seed=42)
        self.assertGreater(len(train_rows), 0)
        self.assertGreater(len(eval_rows), 0)
        ratio = len(train_rows) / len(pairs)
        self.assertTrue(0.75 <= ratio <= 0.85)

    def test_invalid_split(self):
        with self.assertRaises(ValueError):
            split_pairs([{'instruction': 'i', 'response': 'r'}], train_fraction=0.8, seed=42)

    def test_adapter_specs(self):
        specs = _build_adapter_specs()
        self.assertEqual([spec.name for spec in specs], ['A', 'B', 'C'])
        self.assertEqual(specs[0].rank, 8)
        self.assertEqual(specs[1].alpha, 32)
        self.assertIn('o_proj', specs[2].target_modules)


TEST_SUITE = unittest.TestLoader().loadTestsFromTestCase(TestAssignment1BPartB)
print(f'Prepared {TEST_SUITE.countTestCases()} unit tests.')

## 7) Run Tests in VS Code and Review Output

Run the in-notebook unit tests and inspect failures before training/evaluation runs.

In [ ]:
test_result = unittest.TextTestRunner(verbosity=2).run(TEST_SUITE)
assert test_result.wasSuccessful(), 'Unit tests failed. Fix failures before proceeding.'
print('All notebook unit tests passed.')

## 8) Execute Example Scenarios and Inspect Results

Run B1 by default. Enable B2/B3 flags only after dependencies and GPU runtime are ready.

In [ ]:
# B1 execution (enabled by default)
if RUN_B1:
    counts = prepare_b1_dataset(
        clean_dir=CLEAN_CORPUS,
        output_dir=OUTPUTS,
        max_pairs=MAX_PAIRS,
        train_fraction=TRAIN_FRACTION,
        seed=SEED,
    )
    print('B1 complete:')
    print(json.dumps(counts, indent=2))
else:
    print('B1 skipped (RUN_B1=False).')


# B2 execution (opt-in): requires datasets, peft, trl, bitsandbytes, accelerate
if RUN_B2:
    required_paths = [TRAIN_JSONL, EVAL_JSONL]
    missing = [str(p) for p in required_paths if not p.exists()]
    if missing:
        raise FileNotFoundError(f'Missing dataset split files for B2: {missing}. Run B1 first.')

    train_rows = [json.loads(line) for line in TRAIN_JSONL.read_text(encoding='utf-8').splitlines() if line.strip()]
    eval_rows = [json.loads(line) for line in EVAL_JSONL.read_text(encoding='utf-8').splitlines() if line.strip()]

    ADAPTER_ROOT.mkdir(parents=True, exist_ok=True)
    trained_adapters: dict[str, str] = {}
    for spec in _build_adapter_specs():
        out_dir = ADAPTER_ROOT / f'adapter_{spec.name}'
        saved_dir = train_qlora_adapter(
            model_id=MODEL_ID,
            train_rows=train_rows,
            eval_rows=eval_rows,
            adapter_spec=spec,
            output_dir=out_dir,
            max_steps=60,
        )
        trained_adapters[spec.name] = str(saved_dir)

    (OUTPUTS / 'trained_adapters.json').write_text(json.dumps(trained_adapters, indent=2), encoding='utf-8')
    print('B2 complete:')
    print(json.dumps(trained_adapters, indent=2))
else:
    print('B2 skipped (RUN_B2=False).')


# B3 execution (opt-in): compares baseline with adapters A/B/C on the same 3 prompts
if RUN_B3:
    baseline_rows = load_baseline_outputs(BASELINE_PATH)
    adapter_dirs = {spec.name: ADAPTER_ROOT / f'adapter_{spec.name}' for spec in _build_adapter_specs()}
    comparison = compare_adapters(
        model_id=MODEL_ID,
        adapter_dirs=adapter_dirs,
        prompts=DOMAIN_PROMPTS,
        baseline_outputs=baseline_rows,
    )
    if comparison:
        with COMPARISON_CSV.open('w', newline='', encoding='utf-8') as csv_file:
            writer = csv.DictWriter(csv_file, fieldnames=list(comparison[0]))
            writer.writeheader()
            writer.writerows(comparison)
    COMPARISON_JSON.write_text(json.dumps(comparison, ensure_ascii=False, indent=2), encoding='utf-8')
    display(comparison)

    verdict = {
        'best_adapter': 'Manual review required',
        'note': 'Choose the adapter with best domain relevance and factual quality among A/B/C outputs.'
    }
    (OUTPUTS / 'adapter_verdict.json').write_text(json.dumps(verdict, indent=2), encoding='utf-8')
    print('B3 complete. Saved comparison files and verdict placeholder.')
else:
    print('B3 skipped (RUN_B3=False).')